In [1]:
import tensorflow as tf
from tensorflow.keras.callbacks import TensorBoard

In [2]:
from tensorflow.keras.models import load_model
import pickle 
import pandas as pd 
import numpy as np


In [4]:
#Load Tensorboard extension 

%load_ext tensorboard

In [6]:
#Launch Tensorboard session

%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 37616), started 0:00:26 ago. (Use '!kill 37616' to kill it.)

In [3]:
#load pickle files 

#load trained model
model=load_model('model.h5')

In [4]:
#load encoder & scaler 
with open('geo_encoder.pkl','rb') as file:
    label_encoder_geo=pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender=pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler=pickle.load(file)        

In [17]:
#Example input data
input_data={
    'CreditScore':600,
    'Geography':'France',
    'Gender':'Male',
    'Age':40,
    'Tenure':3,
    'Balance':60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':50000
}

In [18]:
geo_encoded=label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=label_encoder_geo.get_feature_names_out(['Geography']))

d:\Work\DL_projects\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [19]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [20]:
#convert to df
input_data_df=pd.DataFrame([input_data])
input_data_df['Gender']=label_encoder_gender.transform(input_data_df['Gender'])

In [21]:
input_data_df=pd.concat([input_data_df.drop("Geography",axis=1),geo_encoded_df],axis=1)
input_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [22]:
#scaling input data
scaled_input_data=scaler.transform(input_data_df)
scaled_input_data

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [26]:
#Predict
prediction=model.predict(scaled_input_data)
predict_probability=prediction[0][0]
if predict_probability>0.5:
    print(f'{predict_probability} Customer is likely to churn')
else:
    print(f'{predict_probability} Customer is not likely to churn')    


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
0.06547018140554428 Customer is not likely to churn
